In [1]:
import os
os.chdir("../")
print(os.getcwd())

c:\Users\unaif\TFM PYTHON\Repositorio\TFM


In [2]:

from src.SIMCA import SIMCA_Multiclass, SIMCA
from src.rgb import img_rgb, imagen_prediccion
from src.preprocesado import snv, savgol
print(os.getcwd())
import pickle

from sklearn.model_selection import train_test_split



c:\Users\unaif\TFM PYTHON\Repositorio\TFM


In [3]:
with open("DATA/arena_binned_sliced.pkl", "rb") as f:
    arena_plasticos = pickle.load(f)
arena_plasticosX=arena_plasticos.labelled_spectra
arena_plasticosY=arena_plasticos.ylabels

with open("DATA/gorliz_binned_sliced.pkl", "rb") as f:
    gorliz_plasticos = pickle.load(f)
gorliz_plasticosX=gorliz_plasticos.labelled_spectra
gorliz_plasticosY=gorliz_plasticos.ylabels
with open("DATA/sonabia_binned_sliced.pkl", "rb") as f:
    sonabia_plasticos = pickle.load(f)
sonabia_plasticosX=sonabia_plasticos.labelled_spectra
sonabia_plasticosY=sonabia_plasticos.ylabels
with open("DATA/sopela_binned_sliced.pkl", "rb") as f:
    sopela_plasticos = pickle.load(f)
sopela_plasticosX=sopela_plasticos.labelled_spectra
sopela_plasticosY=sopela_plasticos.ylabels

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from chemotools.derivative import SavitzkyGolay
from chemotools.feature_selection import RangeCut
from chemotools.scatter import StandardNormalVariate
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import VotingClassifier
from itertools import product


SIMCA_PE=SIMCA(_class=1)
SIMCA_PP=SIMCA(_class=2)
SIMCA_PS=SIMCA(_class=3)

ranged=range(1,20,4)
param={"""simca__n_components""":ranged}


def optimize_SIMCA_SAVGOL_SNV(simca_model,X_train, y_train):

    pipeline = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        simca_model
    )

    modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5
    )

    modelo_.fit(X_train, y_train)
    best_params_dict=modelo_.best_params_
    return best_params_dict


In [5]:
def optimize_SIMCA_SAVGOL(simca_model,X_train, y_train):

    pipeline = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        
        StandardScaler(with_std=False),
        simca_model
    )

    modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5
    )

    modelo_.fit(X_train, y_train)
    best_params_dict=modelo_.best_params_
    return best_params_dict
def optimize_SIMCA_SAVGOL_SNV(simca_model,X_train, y_train):

    pipeline = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        simca_model
    )

    modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5
    )

    modelo_.fit(X_train, y_train)
    best_params_dict=modelo_.best_params_
    return best_params_dict
def optimize_SNV(simca_model,X_train, y_train):
    pipeline = make_pipeline(
        
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        simca_model
    )
    modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5
    )
    modelo_.fit(X_train, y_train)
    best_params_dict=modelo_.best_params_
    return best_params_dict

In [6]:
import numpy as np
def sep_clases(y_train):

    y_train_PE = (y_train == 1).astype(int)
    y_train_PP = (y_train == 2).astype(int)*2
    y_train_PS = (y_train == 3).astype(int)*3
    return y_train_PE, y_train_PP, y_train_PS

In [7]:
pipeline_multi_SAVGOL_SNV = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        multi_simca)
pipeline_multi_SAVGOL = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        StandardScaler(with_std=False),
        multi_simca)
pipeline_multi_SNV = make_pipeline(
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        multi_simca)

NameError: name 'multi_simca' is not defined

In [ ]:
pipeline_multi.predict(X_train)[1]

array([1, 0, 0])

In [8]:
import sklearn.metrics as met
def metrica(
    X,
    y_true,
    modelo,
    playa_test,
    playa_train,
    modo,
    number_component,
    clases
):

    y_pred = modelo.predict(X)

    resultados = []

    for i, clase in enumerate(modelo.classes_):

        y_true_clase = (y_true == clase).astype(int)
        y_pred_clase = y_pred[:, i]

        resultados.append({
            "Playa_train": playa_train,
            "Playa_test": playa_test,
            "Preprocesado": modo,

            "PE_components": number_component[1],
            "PP_components": number_component[2],
            "PS_components": number_component[3],

            "Clase": clases[i],

            "Accuracy": met.accuracy_score(
                y_true_clase,
                y_pred_clase
            ),

            "F1": met.f1_score(
                y_true_clase,
                y_pred_clase
            ),

            "Precision": met.precision_score(
                y_true_clase,
                y_pred_clase
            ),

            "Recall": met.recall_score(
                y_true_clase,
                y_pred_clase
            )
        })

    return pd.DataFrame(resultados)

In [9]:
import pandas as pd


In [10]:
def trainer(xtrain, y_train, modo, playa_trained):

    y_train_PE = (y_train == 1).astype(int)
    y_train_PP = (y_train == 2).astype(int) * 2
    y_train_PS = (y_train == 3).astype(int) * 3

    SIMCA_PE = SIMCA(_class=1)
    SIMCA_PP = SIMCA(_class=2)
    SIMCA_PS = SIMCA(_class=3)

    match modo:

        case "SNV":

            best_params_PE = optimize_SNV(
                SIMCA_PE, xtrain, y_train_PE
            )
            best_params_PP = optimize_SNV(
                SIMCA_PP, xtrain, y_train_PP
            )
            best_params_PS = optimize_SNV(
                SIMCA_PS, xtrain, y_train_PS
            )

            pipeline_multi = make_pipeline(
                StandardNormalVariate(),
                StandardScaler(with_std=False),
                SIMCA_Multiclass(
                    n_components={
                        1: best_params_PE["simca__n_components"],
                        2: best_params_PP["simca__n_components"],
                        3: best_params_PS["simca__n_components"]
                    },
                    mode="soft"
                )
            )

        case "SavGol":

            best_params_PE = optimize_SIMCA_SAVGOL(
                SIMCA_PE, xtrain, y_train_PE
            )
            best_params_PP = optimize_SIMCA_SAVGOL(
                SIMCA_PP, xtrain, y_train_PP
            )
            best_params_PS = optimize_SIMCA_SAVGOL(
                SIMCA_PS, xtrain, y_train_PS
            )

            pipeline_multi = make_pipeline(
                SavitzkyGolay(
                    window_length=5,
                    polyorder=2,
                    deriv=1
                ),
                StandardScaler(with_std=False),
                SIMCA_Multiclass(
                    n_components={
                        1: best_params_PE["simca__n_components"],
                        2: best_params_PP["simca__n_components"],
                        3: best_params_PS["simca__n_components"]
                    },
                    mode="soft"
                )
            )

        case "SavGolSNV":

            best_params_PE = optimize_SIMCA_SAVGOL_SNV(
                SIMCA_PE, xtrain, y_train_PE
            )
            best_params_PP = optimize_SIMCA_SAVGOL_SNV(
                SIMCA_PP, xtrain, y_train_PP
            )
            best_params_PS = optimize_SIMCA_SAVGOL_SNV(
                SIMCA_PS, xtrain, y_train_PS
            )

            pipeline_multi = make_pipeline(
                StandardNormalVariate(),
                SavitzkyGolay(
                    window_length=5,
                    polyorder=2,
                    deriv=1
                ),
                StandardScaler(with_std=False),
                SIMCA_Multiclass(
                    n_components={
                        1: best_params_PE["simca__n_components"],
                        2: best_params_PP["simca__n_components"],
                        3: best_params_PS["simca__n_components"]
                    },
                    mode="soft"
                )
            )

        case _:
            raise ValueError(f"Modo desconocido: {modo}")

    pipeline_multi.fit(xtrain, y_train)
    number_component = {
        1: best_params_PE["simca__n_components"],
        2: best_params_PP["simca__n_components"],
        3: best_params_PS["simca__n_components"]
    }
    resultados = pd.concat([
        metrica(
            arena_plasticosX,
            arena_plasticosY,
            pipeline_multi,
            "Arena",
            playa_trained,
            modo,
            number_component,
            ["PE", "PP", "PS"]
        ),

        metrica(
            gorliz_plasticosX,
            gorliz_plasticosY,
            pipeline_multi,
            "Gorliz",
            playa_trained,
            modo,
            number_component,
            ["PE", "PP", "PS"]
        ),

        metrica(
            sonabia_plasticosX,
            sonabia_plasticosY,
            pipeline_multi,
            "Sonabia",
            playa_trained,
            modo,
            number_component,
            ["PE", "PP", "PS"]
        ),

        metrica(
            sopela_plasticosX,
            sopela_plasticosY,
            pipeline_multi,
            "Sopela",
            playa_trained,
            modo,
            number_component,
            ["PE", "PP", "PS"]
        )
    ], ignore_index=True)

    return resultados

In [12]:
df_arena_snv=trainer(xtrain=arena_plasticosX, y_train=arena_plasticosY, modo="SNV",playa_trained="Arenas" )
df_gorliz_snv=trainer(xtrain=gorliz_plasticosX, y_train=gorliz_plasticosY, modo="SNV",playa_trained="Gorliz" )
df_sonabia_snv=trainer(xtrain=sonabia_plasticosX, y_train=sonabia_plasticosY, modo="SNV",playa_trained="Sonabia" )
df_sopela_snv=trainer(xtrain=sopela_plasticosX, y_train=sopela_plasticosY, modo="SNV",playa_trained="Sopela" )

SIMCA for class 1 trained
with 17 components
SIMCA for class 2 trained
with 17 components
SIMCA for class 3 trained
with 13 components
SIMCA for class 1 trained
with 1 components
SIMCA for class 2 trained
with 13 components
SIMCA for class 3 trained
with 13 components
SIMCA for class 1 trained
with 17 components
SIMCA for class 2 trained
with 17 components
SIMCA for class 3 trained
with 13 components
SIMCA for class 1 trained
with 13 components
SIMCA for class 2 trained
with 17 components
SIMCA for class 3 trained
with 17 components


In [13]:
df_arena_SavGol=trainer(xtrain=arena_plasticosX, y_train=arena_plasticosY, modo="SavGol",playa_trained="Arenas" )
df_gorliz_SavGol=trainer(xtrain=gorliz_plasticosX, y_train=gorliz_plasticosY, modo="SavGol",playa_trained="Gorliz" )
df_sonabia_SavGol=trainer(xtrain=sonabia_plasticosX, y_train=sonabia_plasticosY, modo="SavGol",playa_trained="Sonabia" )
df_sopela_SavGol=trainer(xtrain=sopela_plasticosX, y_train=sopela_plasticosY, modo="SavGol",playa_trained="Sopela" )

SIMCA for class 1 trained
with 17 components
SIMCA for class 2 trained
with 17 components
SIMCA for class 3 trained
with 17 components
SIMCA for class 1 trained
with 17 components
SIMCA for class 2 trained
with 17 components
SIMCA for class 3 trained
with 17 components
SIMCA for class 1 trained
with 9 components
SIMCA for class 2 trained
with 17 components
SIMCA for class 3 trained
with 9 components
SIMCA for class 1 trained
with 13 components
SIMCA for class 2 trained
with 13 components
SIMCA for class 3 trained
with 13 components


In [14]:
df_arena_SavGolSNV=trainer(xtrain=arena_plasticosX, y_train=arena_plasticosY, modo="SavGolSNV",playa_trained="Arenas" )
df_gorliz_SavGolSNV=trainer(xtrain=gorliz_plasticosX, y_train=gorliz_plasticosY, modo="SavGolSNV",playa_trained="Gorliz" )
df_sonabia_SavGolSNV=trainer(xtrain=sonabia_plasticosX, y_train=sonabia_plasticosY, modo="SavGolSNV",playa_trained="Sonabia" )
df_sopela_SavGolSNV=trainer(xtrain=sopela_plasticosX, y_train=sopela_plasticosY, modo="SavGolSNV",playa_trained="Sopela" )

SIMCA for class 1 trained
with 1 components
SIMCA for class 2 trained
with 1 components
SIMCA for class 3 trained
with 17 components
SIMCA for class 1 trained
with 1 components
SIMCA for class 2 trained
with 1 components
SIMCA for class 3 trained
with 17 components
SIMCA for class 1 trained
with 1 components
SIMCA for class 2 trained
with 1 components
SIMCA for class 3 trained
with 1 components
SIMCA for class 1 trained
with 9 components
SIMCA for class 2 trained
with 1 components
SIMCA for class 3 trained
with 5 components


In [ ]:
----

In [15]:
df_final=pd.concat([df_arena_snv, df_gorliz_snv, df_sonabia_snv,df_sopela_snv,df_arena_SavGol,df_gorliz_SavGol,df_sonabia_SavGol, df_sopela_SavGol, df_arena_SavGolSNV, df_gorliz_SavGolSNV, df_sonabia_SavGolSNV, df_sopela_SavGolSNV])
df_final

,Playa_train,Playa_test,Preprocesado,PE_components,PP_components,PS_components,Clase,Accuracy,F1,Precision,Recall
0,Arenas,Arena,SNV,17,17,13,PE,0.768580,0.757034,0.609166,0.999699
1,Arenas,Arena,SNV,17,17,13,PP,0.956276,0.941872,0.895502,0.993307
2,Arenas,Arena,SNV,17,17,13,PS,0.974504,0.955984,0.933772,0.979279
3,Arenas,Gorliz,SNV,17,17,13,PE,0.724978,0.753262,0.604187,1.000000
4,Arenas,Gorliz,SNV,17,17,13,PP,0.950005,0.949966,0.969882,0.930852
...,...,...,...,...,...,...,...,...,...,...,...
7,Sopela,Sonabia,SavGolSNV,9,1,5,PP,0.631948,0.485943,0.376800,0.684096
8,Sopela,Sonabia,SavGolSNV,9,1,5,PS,0.960295,0.924376,0.865043,0.992447
9,Sopela,Sopela,SavGolSNV,9,1,5,PE,0.980446,0.976786,0.954779,0.999832
10,Sopela,Sopela,SavGolSNV,9,1,5,PP,0.526498,0.498573,0.454691,0.551830


In [16]:
df_final

,Playa_train,Playa_test,Preprocesado,PE_components,PP_components,PS_components,Clase,Accuracy,F1,Precision,Recall
0,Arenas,Arena,SNV,17,17,13,PE,0.768580,0.757034,0.609166,0.999699
1,Arenas,Arena,SNV,17,17,13,PP,0.956276,0.941872,0.895502,0.993307
2,Arenas,Arena,SNV,17,17,13,PS,0.974504,0.955984,0.933772,0.979279
3,Arenas,Gorliz,SNV,17,17,13,PE,0.724978,0.753262,0.604187,1.000000
4,Arenas,Gorliz,SNV,17,17,13,PP,0.950005,0.949966,0.969882,0.930852
...,...,...,...,...,...,...,...,...,...,...,...
7,Sopela,Sonabia,SavGolSNV,9,1,5,PP,0.631948,0.485943,0.376800,0.684096
8,Sopela,Sonabia,SavGolSNV,9,1,5,PS,0.960295,0.924376,0.865043,0.992447
9,Sopela,Sopela,SavGolSNV,9,1,5,PE,0.980446,0.976786,0.954779,0.999832
10,Sopela,Sopela,SavGolSNV,9,1,5,PP,0.526498,0.498573,0.454691,0.551830


In [ ]:
#Este metodo no es tan util ya que es facil sobre entrenar o no entrenar suficiente los modelos. Es mejor elegir el numero de componentes a mano, pero queria probar se podria hacer automaticamente.

In [17]:
df_final.to_csv("SIMCA results.csv",";")

C:\Users\unaif\AppData\Local\Temp\ipykernel_16460\2998162889.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  df_final.to_csv("SIMCA results.csv",";")
